# Amazon Food Review Classification - Optimized for Balanced F1/Recall

Goal: Maximize F1 and Recall across ALL 5 classes (not just overall accuracy).

**Strategy:**
- Strong focal loss to focus on hard/minority examples
- Aggressive class weighting to penalize minority errors
- Label smoothing to prevent overconfident predictions
- Post-training threshold tuning per class

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU detected: {len(gpus)}')
else:
    print('WARNING: No GPU. Training will be slow.')

print(f'TensorFlow: {tf.__version__}')

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense, Dropout, Bidirectional,
    SpatialDropout1D, GlobalMaxPooling1D, GlobalAveragePooling1D,
    Concatenate, BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
DATA_PATH = 'amazon_review.csv'

df = pd.read_csv(DATA_PATH)
print(f'Full dataset shape: {df.shape}')
print(f'\nScore distribution:')
counts = df['Score'].value_counts().sort_index()
print(counts)
print(f'\nImbalance ratio: {counts.max()/counts.min():.1f}x')

In [ ]:
df = df.dropna(subset=['Text', 'Score'])
df['Score'] = df['Score'].astype(int)
df['Text'] = df['Text'].astype(str).str.strip()

df_sampled = pd.concat(
    [cls.sample(frac=0.10, random_state=42) for _, cls in df.groupby('Score', sort=False)],
    ignore_index=True
)

print(f'Sampled: {df_sampled.shape[0]} rows')
print(df_sampled['Score'].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(8, 4))
df_sampled['Score'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Score Distribution (10% Sample)')
plt.xlabel('Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_sampled['Text'] = df_sampled['Text'].apply(clean_text)

In [ ]:
MAX_VOCAB_SIZE = 15000
MAX_SEQUENCE_LENGTH = 200
EMBEDDING_DIM = 128

texts = df_sampled['Text'].values
labels = (df_sampled['Score'] - 1).values
num_classes = 5

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
y = to_categorical(labels, num_classes=num_classes)

print(f'X: {X.shape}, y: {y.shape}')

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}')

# Compute class weights with extra penalty for minority classes
y_train_labels = np.argmax(y_train, axis=1)
cw = compute_class_weight('balanced', classes=np.arange(num_classes), y=y_train_labels)
# Multiply by 1.5x to increase penalty on minority classes further
cw = cw * 1.5
class_weights = dict(enumerate(cw))

print('\nClass weights (1.5x amplified):')
for i in range(num_classes):
    print(f'  Score {i+1}: {class_weights[i]:.3f}')

## 3. Loss Functions

In [ ]:
def focal_loss(gamma=2.5, alpha=0.25):
    """Focal loss with strong focus on hard examples."""
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.pow(1.0 - y_pred, gamma)
        focal = weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(focal, axis=1))
    return loss_fn

def label_smoothing_loss(smoothing=0.1):
    """Cross entropy with label smoothing."""
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_smooth = y_true * (1 - smoothing) + smoothing / num_classes
        return tf.reduce_mean(-tf.reduce_sum(y_smooth * tf.math.log(y_pred), axis=1))
    return loss_fn

def combined_loss(gamma=2.5, alpha=0.25, smoothing=0.1, focal_weight=0.7):
    """Combine focal loss and label smoothing."""
    fl = focal_loss(gamma=gamma, alpha=alpha)
    ls = label_smoothing_loss(smoothing=smoothing)
    def loss_fn(y_true, y_pred):
        return focal_weight * fl(y_true, y_pred) + (1 - focal_weight) * ls(y_true, y_pred)
    return loss_fn

print('Loss functions defined.')

## 4. Build Model

In [ ]:
def build_model():
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,))
    x = Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH)(inputs)
    x = SpatialDropout1D(0.3)(x)

    # Two LSTM layers for deeper representation
    x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2))(x)
    x = Bidirectional(LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.2))(x)

    # Pooling
    avg_pool = GlobalAveragePooling1D()(x)
    max_pool = GlobalMaxPooling1D()(x)
    x = Concatenate()([avg_pool, max_pool])

    # Classification head
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-3))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-3))(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    return Model(inputs=inputs, outputs=outputs)

model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss=combined_loss(gamma=2.5, alpha=0.25, smoothing=0.1, focal_weight=0.7),
    metrics=['accuracy']
)
model.summary()

## 5. Train

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

gap = history.history['accuracy'][-1] - history.history['val_accuracy'][-1]
print(f'\nTrain-Val gap: {gap:.4f}')

## 6. Evaluate with Default Threshold

In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

score_names = ['1 star', '2 stars', '3 stars', '4 stars', '5 stars']

print('=' * 60)
print('CLASSIFICATION REPORT (Default Threshold)')
print('=' * 60)
print(classification_report(y_true_classes, y_pred_classes, target_names=score_names, digits=4))

precision, recall, f1, support = precision_recall_fscore_support(
    y_true_classes, y_pred_classes, average=None
)

metrics_df = pd.DataFrame({
    'Score': score_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print('\n' + '=' * 60)
print('PRECISION & RECALL PER CATEGORY')
print('=' * 60)
print(metrics_df.to_string(index=False))

print(f'\nMacro Average F1: {np.mean(f1):.4f}')
print(f'Weighted Average F1: {np.average(f1, weights=support):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(score_names))
width = 0.35

axes[0].bar(x_pos - width/2, precision, width, label='Precision', color='steelblue', edgecolor='black')
axes[0].bar(x_pos + width/2, recall, width, label='Recall', color='coral', edgecolor='black')
axes[0].set_title('Precision & Recall per Category')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(score_names)
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3, axis='y')

cm = confusion_matrix(y_true_classes, y_pred_classes)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im = axes[1].imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set_title('Confusion Matrix (Normalized)')
plt.colorbar(im, ax=axes[1])
tick_marks = np.arange(num_classes)
axes[1].set_xticks(tick_marks)
axes[1].set_xticklabels(score_names, rotation=45)
axes[1].set_yticks(tick_marks)
axes[1].set_yticklabels(score_names)
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, format(cm_norm[i, j], '.2f'),
                     ha='center', va='center', fontsize=9,
                     color='white' if cm_norm[i, j] > 0.5 else 'black')

plt.tight_layout()
plt.show()

## 7. Threshold Tuning for Balanced F1

Instead of always picking the highest probability class, we can shift thresholds to boost minority class recall at the cost of some precision.

In [ ]:
# Tune per-class thresholds to maximize macro F1
# Start with boosting minority classes by lowering their threshold
score_names = ['1 star', '2 stars', '3 stars', '4 stars', '5 stars']

best_macro_f1 = 0
best_thresholds = None

# Grid search over threshold offsets for minority classes
for offset_2 in np.arange(-0.15, 0.05, 0.02):
    for offset_3 in np.arange(-0.10, 0.05, 0.02):
        for offset_4 in np.arange(-0.10, 0.05, 0.02):
            # Apply offsets: lower threshold = more likely to predict that class
            adjusted = y_pred.copy()
            adjusted[:, 0] += 0.0   # 1 star: no change
            adjusted[:, 1] += offset_2  # 2 stars: boost
            adjusted[:, 2] += offset_3  # 3 stars: boost
            adjusted[:, 3] += offset_4  # 4 stars: boost
            adjusted[:, 4] += 0.0   # 5 stars: no change
            
            # Renormalize
            adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)
            
            pred_classes = np.argmax(adjusted, axis=1)
            _, _, f1, _ = precision_recall_fscore_support(
                y_true_classes, pred_classes, average='macro'
            )
            
            if f1 > best_macro_f1:
                best_macro_f1 = f1
                best_thresholds = (0.0, offset_2, offset_3, offset_4, 0.0)

print(f'Best macro F1 after threshold tuning: {best_macro_f1:.4f}')
print(f'Best offsets: 1*={best_thresholds[0]:.2f}, 2*={best_thresholds[1]:.2f}, '
      f'3*={best_thresholds[2]:.2f}, 4*={best_thresholds[3]:.2f}, 5*={best_thresholds[4]:.2f}')

In [ ]:
# Apply best thresholds
adjusted = y_pred.copy()
adjusted[:, 0] += best_thresholds[0]
adjusted[:, 1] += best_thresholds[1]
adjusted[:, 2] += best_thresholds[2]
adjusted[:, 3] += best_thresholds[3]
adjusted[:, 4] += best_thresholds[4]
adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)

y_pred_tuned = np.argmax(adjusted, axis=1)

print('=' * 60)
print('CLASSIFICATION REPORT (Threshold Tuned)')
print('=' * 60)
print(classification_report(y_true_classes, y_pred_tuned, target_names=score_names, digits=4))

precision_t, recall_t, f1_t, support_t = precision_recall_fscore_support(
    y_true_classes, y_pred_tuned, average=None
)

metrics_df_t = pd.DataFrame({
    'Score': score_names,
    'Precision': precision_t,
    'Recall': recall_t,
    'F1-Score': f1_t,
    'Support': support_t
})

print('\n' + '=' * 60)
print('PRECISION & RECALL PER CATEGORY (Threshold Tuned)')
print('=' * 60)
print(metrics_df_t.to_string(index=False))

print(f'\nMacro Average F1: {np.mean(f1_t):.4f}')
print(f'Weighted Average F1: {np.average(f1_t, weights=support_t):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(score_names))
width = 0.35

axes[0].bar(x_pos - width/2, precision_t, width, label='Precision', color='steelblue', edgecolor='black')
axes[0].bar(x_pos + width/2, recall_t, width, label='Recall', color='coral', edgecolor='black')
axes[0].set_title('Precision & Recall (Threshold Tuned)')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(score_names)
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3, axis='y')

cm_t = confusion_matrix(y_true_classes, y_pred_tuned)
cm_t_norm = cm_t.astype('float') / cm_t.sum(axis=1)[:, np.newaxis]
im = axes[1].imshow(cm_t_norm, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set_title('Confusion Matrix (Threshold Tuned)')
plt.colorbar(im, ax=axes[1])
axes[1].set_xticks(tick_marks)
axes[1].set_xticklabels(score_names, rotation=45)
axes[1].set_yticks(tick_marks)
axes[1].set_yticklabels(score_names)
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

for i in range(cm_t.shape[0]):
    for j in range(cm_t.shape[1]):
        axes[1].text(j, i, format(cm_t_norm[i, j], '.2f'),
                     ha='center', va='center', fontsize=9,
                     color='white' if cm_t_norm[i, j] > 0.5 else 'black')

plt.tight_layout()
plt.show()

## 8. Final Comparison

In [ ]:
print('=' * 60)
print('BEFORE vs AFTER THRESHOLD TUNING')
print('=' * 60)

p0, r0, f0, _ = precision_recall_fscore_support(y_true_classes, y_pred_classes, average=None)
p1, r1, f1_scores, _ = precision_recall_fscore_support(y_true_classes, y_pred_tuned, average=None)

comparison = pd.DataFrame({
    'Score': score_names,
    'F1_Before': f0,
    'F1_After': f1_scores,
    'F1_Diff': f1_scores - f0,
    'Recall_Before': r0,
    'Recall_After': r1,
    'Recall_Diff': r1 - r0
})

print(comparison.to_string(index=False))
print(f'\nMacro F1 Before: {np.mean(f0):.4f}')
print(f'Macro F1 After:  {np.mean(f1_scores):.4f}')
print(f'Improvement:     {np.mean(f1_scores) - np.mean(f0):+.4f}')

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')